In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nhattruongdev/musan-noise")

print("Path to dataset files:", path)

100%|██████████| 10.3G/10.3G [04:27<00:00, 41.5MB/s]

Extracting files...


In [ ]:
!pip install librosa soundfile kagglehub tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("pypiahmad/librispeech-asr-corpus")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'librispeech-asr-corpus' dataset.
Path to dataset files: /kaggle/input/librispeech-asr-corpus


In [ ]:
import librosa
import numpy as np

def extract_features(clean, noisy, sr=16000, n_fft=512, hop_length=128):

    clean_stft = librosa.stft(clean, n_fft=n_fft, hop_length=hop_length)
    noisy_stft = librosa.stft(noisy, n_fft=n_fft, hop_length=hop_length)

    clean_mag = np.abs(clean_stft)
    noisy_mag = np.abs(noisy_stft)

    # Ideal Ratio Mask
    mask = clean_mag / (noisy_mag + 1e-8)
    mask = np.clip(mask, 0, 1)

    # Log magnitude as input
    log_noisy_mag = np.log1p(noisy_mag)

    return log_noisy_mag.T, mask.T

In [ ]:
X = []
Y = []

for i in range(200):  # you can increase later
    clean = load_audio(random.choice(clean_files))
    noise = load_audio(random.choice(noise_files))

    snr = random.uniform(0, 10)
    noisy = mix_audio(clean, noise, snr)

    features, mask = extract_features(clean, noisy)

    min_len = min(len(features), len(mask))

    X.append(features[:min_len])
    Y.append(mask[:min_len])

X = np.vstack(X)
Y = np.vstack(Y)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

X shape: (317538, 257)
Y shape: (317538, 257)


In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(257,)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(257, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss='mse'
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_10 (Dense)                │ (None, 512)            │       132,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 257)            │       131,841 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 530,689 (2.02 MB)

 Trainable params: 528,641 (2.02 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [ ]:
history = model.fit(
    X, Y,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Epoch 1/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 87s 19ms/step - loss: 0.1082 - val_loss: 0.0971
Epoch 2/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 85s 19ms/step - loss: 0.0813 - val_loss: 0.0935
Epoch 3/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 97s 22ms/step - loss: 0.0776 - val_loss: 0.0906
Epoch 4/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 94s 21ms/step - loss: 0.0750 - val_loss: 0.0943
Epoch 5/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 88s 20ms/step - loss: 0.0734 - val_loss: 0.0906
Epoch 6/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 84s 19ms/step - loss: 0.0720 - val_loss: 0.0905
Epoch 7/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 86s 19ms/step - loss: 0.0709 - val_loss: 0.0895
Epoch 8/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 87s 19ms/step - loss: 0.0703 - val_loss: 0.0908
Epoch 9/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 85s 19ms/step - loss: 0.0696 - val_loss: 0.0889
Epoch 10/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 139s 18ms/step - loss: 0.0688 - val_loss: 0.0893
Epoch 11/15
4466/4466 ━━━━━━━━━━━━━━━━━━━━ 85s 19ms/step - loss: 0.0683 - val_loss: 0.0889
Epoch 1

In [ ]:
def enhance_audio(model, noisy, sr=16000, n_fft=512, hop_length=128):

    noisy_stft = librosa.stft(noisy, n_fft=n_fft, hop_length=hop_length)
    noisy_mag = np.abs(noisy_stft)
    noisy_phase = np.angle(noisy_stft)

    log_noisy_mag = np.log1p(noisy_mag).T

    predicted_mask = model.predict(log_noisy_mag, verbose=0)

    enhanced_mag = noisy_mag * predicted_mask.T

    enhanced_stft = enhanced_mag * np.exp(1j * noisy_phase)
    enhanced_audio = librosa.istft(enhanced_stft, hop_length=hop_length)

    # Normalize safely
    enhanced_audio = enhanced_audio / (np.max(np.abs(enhanced_audio)) + 1e-8)

    return enhanced_audio

In [ ]:
test_clean = load_audio(random.choice(clean_files))
test_noise = load_audio(random.choice(noise_files))

test_noisy = mix_audio(test_clean, test_noise, 5)

enhanced = enhance_audio(model, test_noisy)

import soundfile as sf
sf.write("noisy.wav", test_noisy, 16000)
sf.write("enhanced.wav", enhanced, 16000)